<a href="https://colab.research.google.com/github/satyamroy3913-maker/GenAi_Task/blob/main/Day01_Goal03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os, sys, re, time, json, subprocess, getpass
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "pydantic", "pandas"], check=True)

import pandas as pd
from google import genai

DELAY = 6.5   # free tier ke liye. Paid key ho to 0.5

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
api_key = api_key or os.environ.get("GEMINI_API_KEY") or getpass.getpass("Gemini API key paste karo: ")
client = genai.Client(api_key=api_key)

CANDIDATES = ["gemini-flash-lite-latest", "gemini-2.5-flash-lite", "gemini-flash-latest", "gemini-2.5-flash"]

def pick_working_model(candidates):
    for name in candidates:
        for attempt in range(2):
            try:
                r = client.models.generate_content(model=name, contents="Reply with the word OK.")
                if r.text:
                    return name
            except Exception as e:
                print(f"  {name}: {str(e)[:70]}... (try {attempt+1}/2)")
                time.sleep(8)
    return None

MODEL = pick_working_model(CANDIDATES)
if MODEL is None:
    raise RuntimeError("Abhi koi model respond nahi kar raha. 15-20 min baad dobara run karo.")
print("Using model:", MODEL)

Gemini API key paste karo: ··········
Using model: gemini-flash-lite-latest


In [ ]:
SELF_DIR = "/content/bootcamp-repo/self_repair"
INPUT_DIR = os.path.join(SELF_DIR, "inputs")
os.makedirs(INPUT_DIR, exist_ok=True)
print("Folders ready:", SELF_DIR)

Folders ready: /content/bootcamp-repo/self_repair


In [ ]:
%%writefile /content/bootcamp-repo/self_repair/schema.py
import re
from typing import Optional
from pydantic import BaseModel, Field, field_validator


class Address(BaseModel):
    city: Optional[str] = Field(default=None, description="City name")
    pincode: Optional[str] = Field(default=None, description="Postal code")

    @field_validator("pincode")
    @classmethod
    def check_pincode(cls, v):
        if v is not None and not re.fullmatch(r"\d{6}", v):
            raise ValueError("pincode must be exactly 6 digits")
        return v


class ContactCard(BaseModel):
    name: str
    email: Optional[str]
    phone: Optional[str]
    address: Address

    @field_validator("name")
    @classmethod
    def check_name(cls, v):
        if not v.strip():
            raise ValueError("name must not be empty")
        return v.strip()

    @field_validator("email")
    @classmethod
    def check_email(cls, v):
        if v is not None and not re.fullmatch(r"[^@\s]+@[^@\s]+\.[^@\s]+", v):
            raise ValueError("email must be a valid email address like name@domain.com")
        return v

    @field_validator("phone")
    @classmethod
    def check_phone(cls, v):
        if v is not None and not re.fullmatch(r"\d{10}", v):
            raise ValueError("phone must be exactly 10 digits (no spaces, symbols, country code or leading 0)")
        return v

Writing /content/bootcamp-repo/self_repair/schema.py


In [ ]:
%%writefile /content/bootcamp-repo/self_repair/repair_loop.py
import time
from pydantic import ValidationError
from google.genai import types
from schema import ContactCard

# Assumed price (USD per 1M tokens). Apne model ka current price Google AI pricing page se check karke badal lo.
PRICE_IN = 0.10
PRICE_OUT = 0.40

BASE_PROMPT = """Extract the contact details from the messy text below into the required JSON schema.
Use null for anything that is not present. Do not invent information.

TEXT:
{text}"""

REPAIR_PROMPT = """{base}

Your previous answer was:
{prev}

It failed validation with this error:
{err}

Fix the problems and return the corrected JSON only. Use null if the correct value cannot be found in the text."""


def _generate(client, model, prompt, api_retries=8):
    """API call. 503/429 jaise server errors par alag se retry (ye 'repair retries' me nahi gine jaate)."""
    for attempt in range(api_retries):
        try:
            return client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=ContactCard,
                    temperature=0,
                ),
            )
        except Exception as e:
            wait = min(15 * (attempt + 1), 90)
            print(f"  API error ({str(e)[:70]}...) - {wait}s baad retry")
            time.sleep(wait)
    raise RuntimeError("Model abhi busy hai. 10-15 min baad dobara run karo.")


def _cost(tokens_in, tokens_out):
    return (tokens_in * PRICE_IN + tokens_out * PRICE_OUT) / 1_000_000


def _short_error(e: ValidationError):
    return "; ".join(f"{'.'.join(str(x) for x in err['loc'])}: {err['msg']}" for err in e.errors())


def repair_loop(client, model, text, max_retries=3, delay=0):
    base = BASE_PROMPT.format(text=text)
    prompt = base
    errors, total_in, total_out, first_cost = [], 0, 0, 0.0

    for attempt in range(max_retries + 1):          # 1 pehli try + max 3 retries
        resp = _generate(client, model, prompt)
        raw = resp.text or ""
        u = resp.usage_metadata
        t_in = (u.prompt_token_count or 0) if u else 0
        t_out = ((u.candidates_token_count or 0) + (getattr(u, "thoughts_token_count", 0) or 0)) if u else 0
        total_in += t_in
        total_out += t_out
        if attempt == 0:
            first_cost = _cost(t_in, t_out)
        if delay:
            time.sleep(delay)

        try:
            card = ContactCard.model_validate_json(raw)
            return {"card": card.model_dump(), "first_try_valid": attempt == 0, "final_valid": True,
                    "num_retries": attempt, "errors_seen": errors,
                    "total_cost": _cost(total_in, total_out), "first_cost": first_cost, "last_raw": raw}
        except ValidationError as e:
            err = _short_error(e)
            errors.append(err)
            prompt = REPAIR_PROMPT.format(base=base, prev=raw, err=err)

    return {"card": None, "first_try_valid": False, "final_valid": False,
            "num_retries": max_retries, "errors_seen": errors,
            "total_cost": _cost(total_in, total_out), "first_cost": first_cost, "last_raw": raw}

Writing /content/bootcamp-repo/self_repair/repair_loop.py


In [ ]:
import importlib
sys.path.insert(0, SELF_DIR)
import schema, repair_loop as rl
importlib.reload(schema); importlib.reload(rl)
from schema import ContactCard
from repair_loop import repair_loop
from pydantic import ValidationError

good = {"name": "Rahul Mehta", "email": "rahul@acme.in", "phone": "9876543210",
        "address": {"city": "Pune", "pincode": "411001"}}
print("GOOD ->", ContactCard.model_validate(good).model_dump())

bad = {"name": "Rahul", "email": "rahul@@acme", "phone": "+91 98765 43210",
       "address": {"city": "Pune", "pincode": "4110"}}
try:
    ContactCard.model_validate(bad)
except ValidationError as e:
    print("\nBAD ->", rl._short_error(e))

GOOD -> {'name': 'Rahul Mehta', 'email': 'rahul@acme.in', 'phone': '9876543210', 'address': {'city': 'Pune', 'pincode': '411001'}}

BAD -> email: Value error, email must be a valid email address like name@domain.com; phone: Value error, phone must be exactly 10 digits (no spaces, symbols, country code or leading 0); address.pincode: Value error, pincode must be exactly 6 digits


In [ ]:
INPUTS = [
 "RAHUL MEHTA\nSr. Manager - Sales\nrahul.mehta@acmecorp.in\nM: +91 98765 43210\n12 MG Road, Pune 411001",
 "Priya Sharrna\nPriya.sharrna(at)gmail.com\nTel 022-2345 6789 / Mob 9820012345\nFlat 4B, Andheri West, Mumbai - 400053",
 "Amit Verma\nDirector\nwww.vermaconsult.com\nPh: 98OO1 23456\nSector 18, Noida UP 201301",
 "सुनील कुमार\nsunil.kumar@rediffmail.com\n9876543210\nगांधी नगर, जयपुर 302015",
 "Dr. Neha Kapoor MBBS\nneha kapoor @ hospitalmail . com\n+91-98 1111 2222\nBangalore 560034",
 "Karan Singh\nPhone: 99887 76655\nEmail: karan.singh@@techmail.com\nChandigarh 1600l7",
 "Ms. Anjali Desai | Marketing Lead | anjali.desai@brandworks.co.in | 0091 9822334455 | Koregaon Park, Pune",
 "Vikram Rathore\nM +91 7000 123 456\nvikram_rathore@gmial.com\nIndore, M.P. 452001",
 "S. Iyer\nsiyer@infosolutions.in\nCall: 044 2814 5566\nChennai - 6000 28",
 "NAME: Rohan Gupta\nEMAIL: rohan.gupta@startup.io\nPHONE: (+91) 9-8-7-6-5-4-3-2-1-0\nCITY: Delhi PIN: 110 001",
 "Fatima Sheikh\n+91 91234 56789 , +91 98200 11122\nfatima.sheikh@designhub.com\nHyderabad 500016",
 "Tushar Joshi\nChief Technology Officer\ntushar@joshitech.in\nPune 41100",
 "Lakshmi Narayanan\nlakshmi.n@southmail.com\n98 4 0 0 1 2 3 4 5\nKochi, Kerala 682016",
 "Mohd. Imran Khan\nimran.khan@gmail.com\nTel: 9811-223-344 ext 12\nLucknow UP 226001",
 "Deepa Nair\nPhone +1 415 555 0132\ndeepa.nair@globalfirm.com\nSan Francisco CA 94105",
 "ANKIT AGARWAL\nGen. Merchant\nPH 0141-2370098\nMalviya Nagar Jaipur 302017",
 "Pooja Reddy\nVisit us: www.poojaboutique.in\n@poojaboutique\nHyderabad",
 "Hello this is Suresh Patel, you can reach me on nine eight seven six five four three two one zero or mail suresh.patel@mailbox.in, I stay in Surat 395007",
 "Jas win der Kaur\nj.kaur@@pbmail.com\n+91 9 8 1 4 0 0 1 1 2 2\nLudhiana 14l001",
 "~~ M R. R A H U L ~~\nema1l: rahul##corp.com\nph: 123-45\nPune",
]
assert len(INPUTS) == 20

for i, text in enumerate(INPUTS, start=1):
    with open(os.path.join(INPUT_DIR, f"input_{i:02d}.txt"), "w") as f:
        f.write(text)
print("20 inputs saved:", len(os.listdir(INPUT_DIR)))

20 inputs saved: 20


In [ ]:
rows, details = [], []
for i, text in enumerate(INPUTS, start=1):
    res = repair_loop(client, MODEL, text, max_retries=3, delay=DELAY)
    rows.append({
        "input_id": i,
        "first_try_valid": res["first_try_valid"],
        "final_valid": res["final_valid"],
        "num_retries": res["num_retries"],
        "errors_seen": " || ".join(res["errors_seen"]),
        "total_cost": round(res["total_cost"], 6),
    })
    details.append({"input_id": i, "first_cost": res["first_cost"], "card": res["card"]})
    status = "OK 1st try" if res["first_try_valid"] else ("FIXED after %d retry" % res["num_retries"] if res["final_valid"] else "FAILED")
    print(f"[{i:02d}/20] {status}")
    for e in res["errors_seen"]:
        print("     error:", e[:110])

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(SELF_DIR, "results.csv"), index=False)
with open(os.path.join(SELF_DIR, "results_details.json"), "w") as f:
    json.dump(details, f, indent=2, ensure_ascii=False)
results_df

[01/20] FIXED after 1 retry
     error: phone: Value error, phone must be exactly 10 digits (no spaces, symbols, country code or leading 0)
[02/20] OK 1st try
  API error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur...) - 15s baad retry
  API error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur...) - 15s baad retry
[03/20] FIXED after 1 retry
     error: phone: Value error, phone must be exactly 10 digits (no spaces, symbols, country code or leading 0)
[04/20] OK 1st try
  API error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur...) - 15s baad retry
[05/20] FIXED after 1 retry
     error: email: Value error, email must be a valid email address like name@domain.com; phone: Value error, phone must b
  API error (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur...) - 15s baad retry
[06/20] FIXED after 1 retry
     error: email: Value error, email must be a valid email address like name@domai

,input_id,first_try_valid,final_valid,num_retries,errors_seen,total_cost
0,1,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000102
1,2,True,True,0,,0.000043
2,3,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000080
3,4,True,True,0,,0.000039
4,5,False,True,1,"email: Value error, email must be a valid emai...",0.000093
5,6,False,True,1,"email: Value error, email must be a valid emai...",0.000080
6,7,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000087
7,8,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000087
8,9,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000088
9,10,False,True,1,"phone: Value error, phone must be exactly 10 d...",0.000098


In [ ]:
from collections import Counter

n = len(results_df)
first_ok = int(results_df["first_try_valid"].sum())
final_ok = int(results_df["final_valid"].sum())
fixed = final_ok - first_ok
failed = n - final_ok
cost_with = results_df["total_cost"].sum()
cost_without = sum(d["first_cost"] for d in details)

# error types: pehle error segment ka field naam
ctr = Counter()
for s in results_df["errors_seen"]:
    for attempt_err in [x for x in s.split(" || ") if x]:
        for seg in attempt_err.split("; "):
            field = seg.split(":")[0].strip() or "json_format"
            ctr[field] += 1

# kaun se inputs repair se theek hue
fixed_ids = results_df[(~results_df["first_try_valid"]) & (results_df["final_valid"])]["input_id"].tolist()
failed_ids = results_df[~results_df["final_valid"]]["input_id"].tolist()

def per_success(cost, ok):
    return f"${cost/ok:.6f}" if ok else "n/a"

md = ["# Self-Repair Comparison\n",
      f"Model: `{MODEL}` | Inputs: {n} | Max retries: 3 | Temperature: 0\n",
      "## Success rate\n",
      "| Mode | Valid outputs | Success rate | Estimated cost (USD) | Cost per valid output |",
      "|---|---|---|---|---|",
      f"| Without repair (single attempt) | {first_ok}/{n} | {first_ok/n*100:.0f}% | ${cost_without:.6f} | {per_success(cost_without, first_ok)} |",
      f"| With repair (up to 3 retries) | {final_ok}/{n} | {final_ok/n*100:.0f}% | ${cost_with:.6f} | {per_success(cost_with, final_ok)} |",
      "",
      f"- Succeeded on first try: **{first_ok}**",
      f"- Fixed after retry: **{fixed}** (input ids: {fixed_ids or 'none'})",
      f"- Failed all attempts: **{failed}** (input ids: {failed_ids or 'none'})",
      "",
      "_Single-attempt numbers come from the first call of the same run (temperature 0), so both modes use identical first outputs._",
      "_Cost is an estimate from token counts using assumed per-token prices; check the current price list._\n",
      "## Common error types seen (count of times a field failed validation)\n",
      "| Field | Times failed |", "|---|---|"]
for k, v in ctr.most_common():
    md.append(f"| {k} | {v} |")
if not ctr:
    md.append("| none | 0 |")

md += ["\n## Cost vs success rate trade-off\n",
       f"Repair raised success from {first_ok/n*100:.0f}% to {final_ok/n*100:.0f}% "
       f"for an extra ${cost_with - cost_without:.6f} on {n} inputs "
       f"({(cost_with/cost_without - 1)*100 if cost_without else 0:.0f}% more cost). "
       "Only inputs that fail validation pay for extra calls, so the cost grows with the failure rate.",
       "\nInputs that are truly unrecoverable (information missing or too garbled) fail after all retries; "
       "those burn the most tokens, so a retry cap is important.",
       "\n## My observations (apne words me likho)\n",
       "- Which errors did the retry fix, and why could the model fix them on the second try?",
       "- Which inputs failed even after 3 retries, and why?",
       "- Would you use repair in production? With what retry limit?"]

with open(os.path.join(SELF_DIR, "comparison.md"), "w") as f:
    f.write("\n".join(md))
print("\n".join(md[:14]))

# Self-Repair Comparison

Model: `gemini-flash-lite-latest` | Inputs: 20 | Max retries: 3 | Temperature: 0

## Success rate

| Mode | Valid outputs | Success rate | Estimated cost (USD) | Cost per valid output |
|---|---|---|---|---|
| Without repair (single attempt) | 4/20 | 20% | $0.000781 | $0.000195 |
| With repair (up to 3 retries) | 20/20 | 100% | $0.001539 | $0.000077 |

- Succeeded on first try: **4**
- Fixed after retry: **16** (input ids: [1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 20])
- Failed all attempts: **0** (input ids: none)

_Single-attempt numbers come from the first call of the same run (temperature 0), so both modes use identical first outputs._
_Cost is an estimate from token counts using assumed per-token prices; check the current price list._



In [ ]:
for root, dirs, files_ in os.walk(SELF_DIR):
    if "__pycache__" in root:
        continue
    level = root.replace(SELF_DIR, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for fn in sorted(files_):
        print("  " * (level + 1) + fn)

from google.colab import files
!cd /content/bootcamp-repo && zip -r self_repair.zip self_repair -x "*/__pycache__/*"
files.download("/content/bootcamp-repo/self_repair.zip")

self_repair/
  comparison.md
  repair_loop.py
  results.csv
  results_details.json
  schema.py
  inputs/
    input_01.txt
    input_02.txt
    input_03.txt
    input_04.txt
    input_05.txt
    input_06.txt
    input_07.txt
    input_08.txt
    input_09.txt
    input_10.txt
    input_11.txt
    input_12.txt
    input_13.txt
    input_14.txt
    input_15.txt
    input_16.txt
    input_17.txt
    input_18.txt
    input_19.txt
    input_20.txt
  adding: self_repair/ (stored 0%)
  adding: self_repair/repair_loop.py (deflated 58%)
  adding: self_repair/results.csv (deflated 87%)
  adding: self_repair/inputs/ (stored 0%)
  adding: self_repair/inputs/input_13.txt (deflated 6%)
  adding: self_repair/inputs/input_02.txt (deflated 6%)
  adding: self_repair/inputs/input_05.txt (deflated 9%)
  adding: self_repair/inputs/input_18.txt (deflated 20%)
  adding: self_repair/inputs/input_08.txt (deflated 9%)
  adding: self_repair/inputs/input_07.txt (deflated 9%)
  adding: self_repair/inputs/input_03.tx

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>